In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.patches import ConnectionPatch
import pickle
import pandas as pd
import numpy as np
from matplotlib.backends.backend_pdf import PdfPages
import os

In [ ]:
DATESTR = "2026-08-07"
with open(f'{DATESTR}_plot_test_iter0_synthdata.pkl','rb') as f:
    pkl_data = pickle.load(f)
print(len(pkl_data))

In [ ]:
synth_spec, data, __, weights, loglam = pkl_data
print(synth_spec.shape, data.shape, weights.shape, loglam.shape)

In [ ]:
stars = pd.read_parquet(f"parent_stars_2026-08-05.parquet")
print("num of stars in parquet:", stars.shape[0])

spectra = pd.read_parquet("spectra_iter1_2026-08-07_plot_test.parquet")
print("num of spectra in parquet:", spectra.shape[0])

In [ ]:
spectra.columns.to_list()

In [ ]:
#define constants
c = 299792.458 # km/s
LN10 = np.log(10.)
MAX_IVAR = 2.5e3
MIN_IVAR = 0.1
H_ALPHA, H_BETA = 6564.614, 4862.721 #Reference: from classic.sdss.org
H_GAMMA = 4340.47+2
H_DELTA = 4101.73+2
HE_I_6680 = 6678.15+2 #Reference: Johanna brain
HE_I_4026 = 4026.19+2
HE_I_4471 = 4471.68+2
HE_I_4922 = 4921.93+2
HE_I_4388 = 4387.93+2
HE_I_5878 = 5878.0+2


HE_II_4686 = 4685.68+2 #Reference: Johanna brain
HE_II_4542 = 4541.59+2
HE_II_4200 = 4199.83+2
HE_II_5411 = 5411.52+2

#lines
lines_balmer = [H_ALPHA, H_BETA, H_GAMMA, H_DELTA]
lines_HE_I = [HE_I_6680, HE_I_4026, HE_I_4471, HE_I_4922, HE_I_4388, HE_I_5878]
lines_HE_II = [HE_II_4686, HE_II_4542, HE_II_4200, HE_II_5411]
all_lines = np.concatenate([lines_balmer, lines_HE_I, lines_HE_II])
mask_lines = lines_balmer

#line labes
line_labels_balmer = ["Halpha", "Hbeta", "Hgamma", "Hdelta"] # JMH: add Hgamm, H delta also
line_labels_HE_I = ["HeI_6680", "HeI_4026", "HeI_4471", "HeI_4922", "HeI_4388", "HeI_4713"]
line_labels_HE_II = ["HeII_4686", "HeII_4542", "HeII_4200"]
all_line_labels = np.concatenate([line_labels_balmer, line_labels_HE_I, line_labels_HE_II])

statistics = ["EW", "shift", "width"]
moment_names = ["EW", "M1", "M2", "M3", "M4"]


H_delta_lnlam_line = 1000 / c 
H_delta_loglam_line = H_delta_lnlam_line / LN10 
HE_delta_lnlam_line = 500 / c
HE_delta_loglam_line = HE_delta_lnlam_line / LN10
# log_H_ALPHA, log_H_BETA, log_HE_I, log_HE_II = np.log10(H_ALPHA), np.log10(H_BETA), np.log10(HE_I), np.log10(HE_II)

#delta loglam line for labeling
delta_balmer = np.zeros(len(lines_balmer)) + (1000 / c) / LN10 
delta_HE_I = np.zeros(len(lines_HE_I)) + (500 / c) / LN10 
delta_HE_II = np.zeros(len(lines_HE_II)) + (500 / c) / LN10 
all_delta_loglams = np.concatenate([delta_balmer, delta_HE_I, delta_HE_II])


#indexing is super important for these three arrays 
print(all_lines.shape, all_line_labels.shape, all_delta_loglams.shape)
print(all_lines, all_line_labels, all_delta_loglams)

H_delta_lnlam_line = 1000 / c 
H_delta_loglam_line = H_delta_lnlam_line / LN10 
HE_delta_lnlam_line = 500 / c
HE_delta_loglam_line = HE_delta_lnlam_line / LN10
log_H_ALPHA, log_H_BETA = np.log10(H_ALPHA), np.log10(H_BETA)


half_wids = [H_delta_loglam_line, H_delta_loglam_line, H_delta_loglam_line, H_delta_loglam_line,  HE_delta_loglam_line, HE_delta_loglam_line, HE_delta_loglam_line, HE_delta_loglam_line,
            HE_delta_loglam_line, HE_delta_loglam_line, HE_delta_loglam_line, HE_delta_loglam_line, HE_delta_loglam_line, HE_delta_loglam_line] 

assert len(half_wids) == len(all_lines)

#THIS MUST BE ASCENDING ORDER
temp_cuts = np.array([10000, 15000, 25000, np.inf])
mod_list = [0,1]

#How many iterations?
iterations = 3

In [ ]:
def loglam_halfwidth(velocity_kms):
    """Half-width in log10(wavelength) space corresponding to a velocity."""
    return (velocity_kms / c) / np.log(10.0)
    
def line_span(center_wave, velocity_kms=1000.0):
    """
    Wavelength (lo, hi) bounds spanning +/- velocity_kms/2 around center_wave,
    i.e. a window of total width `velocity_kms`.
    """
    dloglam = loglam_halfwidth(velocity_kms / 2.0)
    loglam0 = np.log10(center_wave)
    return 10 ** (loglam0 - dloglam), 10 ** (loglam0 + dloglam)

In [ ]:
def add_lines(ax, lines, color, label, halfwidth_kms):
    for l, line in enumerate(lines):
        lo, hi = line_span(line, 2 * halfwidth_kms) 
        ax.axvspan(lo, hi, alpha=0.2, color=color)
        if l == 0:
            ax.axvline(line, label=label, lw=1, alpha=0.85, zorder=-10, color=color)
        else:
            ax.axvline(line, lw=1, alpha=0.85, zorder=-10, color=color)


In [ ]:
#Sample plot

f = plt.figure(figsize=(15, 10))
lam = 10 ** loglam

i = 101

teffs = spectra["Teff_fit"].to_numpy()
spec_files = spectra["SPEC_FILE"].to_numpy()

plt.step(lam, data[i], c="k", where="mid")
plt.plot(lam, synth_spec[i],"r-", lw = 1)

for l, line in enumerate(lines_balmer):
    lo, hi = line_span(line, 500)
    plt.axvspan(lo, hi, alpha = 0.2, color = "green")
    if l == 0:
        plt.axvline(line, label = "Balmer lines", lw=1, alpha=0.75, zorder=-10, color = "green")
    else:
        plt.axvline(line, lw=1, alpha=0.75, zorder=-10, color = "green")
        
for l, line in enumerate(lines_HE_I):
    lo, hi = line_span(line, 500)
    plt.axvspan(lo, hi, alpha = 0.2, color = "navy")

    if l == 0: 
        plt.axvline(line, label = "HeI lines", lw=1, alpha=0.75, zorder=-10, color = "navy")
    else:
        plt.axvline(line, lw=1, alpha=0.75, zorder=-10, color = "navy")
        
for l, line in enumerate(lines_HE_II):
    lo, hi = line_span(line, 500)
    plt.axvspan(lo, hi, alpha = 0.2, color = "hotpink")

    if l == 0: 
        plt.axvline(line, label = "HeII lines", lw=1, alpha=0.75, zorder=-10, color = "hotpink")
    else:
        plt.axvline(line, lw=1, alpha=0.75, zorder=-10, color = "hotpink")

plt.legend()    
plt.ylim(0.5,1.4)
plt.xlim(4000, 9000)
plt.xlabel("Wavelength (angstrom)")
plt.ylabel("Normalized Flux")
plt.title(f"{spec_files[i]} and temp {teffs[i]}")
plt.show()
plt.close()


f = plt.figure(figsize = (15,10))
resid =  data - synth_spec
plt.step(lam, resid[i], c="gray", where = "mid")
for l, line in enumerate(lines_balmer):
    lo, hi = line_span(line, 500)
    plt.axvspan(lo, hi, alpha = 0.2, color = "green")
    if l == 0:
        plt.axvline(line, label = "Balmer lines", lw=1, alpha=0.75, zorder=-10, color = "green")
    else:
        plt.axvline(line, lw=1, alpha=0.75, zorder=-10, color = "green")
        
for l, line in enumerate(lines_HE_I):
    lo, hi = line_span(line, 500)
    plt.axvspan(lo, hi, alpha = 0.2, color = "navy")

    if l == 0: 
        plt.axvline(line, label = "HeI lines", lw=1, alpha=0.75, zorder=-10, color = "navy")
    else:
        plt.axvline(line, lw=1, alpha=0.75, zorder=-10, color = "navy")
        
for l, line in enumerate(lines_HE_II):
    lo, hi = line_span(line, 500)
    plt.axvspan(lo, hi, alpha = 0.2, color = "hotpink")

    if l == 0: 
        plt.axvline(line, label = "HeII lines", lw=1, alpha=0.75, zorder=-10, color = "hotpink")
    else:
        plt.axvline(line, lw=1, alpha=0.75, zorder=-10, color = "hotpink")
plt.title("Residual Plot")
plt.legend()
plt.xlim(H_ALPHA - 200, H_ALPHA + 200)
plt.show()
plt.close()

In [ ]:
def connect_zoom(ax_main, ax_zoom, xmin, xmax, edgecolor="0.4", lw=0.8):
    """
    Shade the [xmin, xmax] region in ax_main and draw dashed connector
    lines from its bottom edge down to the top edge of ax_zoom.
    """
    #ax_main.axvspan(xmin, xmax, color=edgecolor, alpha=0.12, zorder=0)
    y_main = ax_main.get_ylim()[0]   # bottom edge of the overview panel
    y_zoom = ax_zoom.get_ylim()[1]   # top edge of the zoom panel
    for x in (xmin, xmax):
        con = ConnectionPatch(
            xyA=(x, y_main), coordsA=ax_main.transData,
            xyB=(x, y_zoom), coordsB=ax_zoom.transData,
            color=edgecolor, lw=lw, ls="--", zorder=0,
        )
        ax_main.figure.add_artist(con)

In [ ]:
#statistics

#trial index
i = 1200
i = 1433
i = 5356
i = 5745
i = 577

teffs = spectra["Teff_fit"].to_numpy()
spec_files = spectra["SPEC_FILE"].to_numpy()
print("teff shape:", teffs.shape)
print("specfile shape:", spec_files.shape)

m1 = spectra["nana_Halpha_M1"].to_numpy()
m1_errs= spectra["nana_Halpha_M1_err"].to_numpy()

ew = spectra["nana_Halpha_EW"].to_numpy()
ew_errs = spectra["nana_Halpha_EW_err"].to_numpy()
print("ew shape:", ew.shape)

m2 = spectra["nana_Halpha_M2"].to_numpy()
m2_errs = spectra["nana_Halpha_M2_err"].to_numpy()

m3 = spectra["nana_Halpha_M3"].to_numpy()
m3_errs = spectra["nana_Halpha_M3_err"].to_numpy()

m4 = spectra["nana_Halpha_M4"].to_numpy()
m4_errs = spectra["nana_Halpha_M4_err"].to_numpy()

bof = spectra["nana_bof"].to_numpy()

gaia_ids = spectra["GAIA_ID"].to_numpy()
print(f"gaia id shape {gaia_ids.shape}")
centroid = m1 / ew                  
linewidth = np.sqrt(m2 / ew)                 
skew = (m3 / ew) / (m2 / ew) ** 1.5
kurtosis = (m4 / ew) / (m2 / ew) ** 2    

print(centroid.shape, linewidth.shape, skew.shape, kurtosis.shape)

In [ ]:
# ---------------- Helper to build one zoom+residual column ----------------
def make_zoom_column(gs_slot, center, halfwidth_ang, title):
    gs_zoom = gridspec.GridSpecFromSubplotSpec(
        2, 1, subplot_spec=gs_slot, height_ratios=[2, 1], hspace=0.08)
    ax_main = fig.add_subplot(gs_zoom[0])
    ax_res = fig.add_subplot(gs_zoom[1], sharex=ax_main)

    ax_main.step(lam, data[i], c="k", where="mid")
    ax_main.plot(lam, synth_spec[i], "r-", lw=1)
    ax_main.fill_between(
    lam,
    data[i] - 1. / np.sqrt(weights[i]),
    data[i] + 1. / np.sqrt(weights[i]),
    color="k", step="mid", alpha=0.1,
    )
    ax_res.step(lam, resid[i], c="k", where="mid")
    ax_res.axhline(0.0, color="0.3", lw=0.6)
    ax_res.fill_between(
    lam,
    resid[i] - 1. / np.sqrt(weights[i]),
    resid[i] + 1. / np.sqrt(weights[i]),
    color="k", step="mid", alpha=0.1,
    )

    for ax in (ax_main, ax_res):
        add_lines(ax, lines_balmer, "palevioletred", None, 1000)
        add_lines(ax, lines_HE_I, "darkseagreen", None, 500)
        add_lines(ax, lines_HE_II, "gold", None, 500)
        ax.set_xlim(center - halfwidth_ang, center + halfwidth_ang)

    ax_main.set_ylim(0.3, 2)
    ax_main.set_title(title)
    plt.setp(ax_main.get_xticklabels(), visible=False)
    ax_res.set_ylabel("Resid.")
    ax_res.set_xlabel("Wavelength (Å)")
    ax_res.set_ylim(-0.3, 1.4)
    ax_res.axhline(0.0, color="red", lw=0.8)
    return ax_main, ax_res

In [ ]:
i = 13426
fig = plt.figure(figsize=(12, 8))
gs = gridspec.GridSpec(2, 3, height_ratios=[1.3, 1.0], hspace=0.3, wspace=0.3)

# ---------------- Top overview panel ----------------
ax_top = fig.add_subplot(gs[0, :])
ax_top.step(lam, data[i], c="k", where="mid", lw=0.75)
ax_top.plot(lam, synth_spec[i], "r-", lw=0.75)

stats_text = (
    f"Teff = {teffs[i]:.0f}K\n"
    f"EW(H$\\alpha$) = {ew[i]:.1f}\n"
    f"Centroid = {centroid[i]:.1f}\n"
    f"Linewidth = {linewidth[i]:.1f}\n"
    f"Skew = {skew[i]:.1f}\n"
    f"Kurtosis = {kurtosis[i]:.1f}\n"
    f"BOF = {bof[i]:.1f}")

ax_top.text(
    0.87, 0.40, stats_text,
    transform=ax_top.transAxes,
    fontsize=10, va="top", ha="left",
    bbox=dict(boxstyle="round", facecolor="white", edgecolor="0.5", alpha=0.85),
)

add_lines(ax_top, lines_balmer, "palevioletred", "Balmer lines", 1000)
add_lines(ax_top, lines_HE_I, "darkseagreen", "HeI lines", 500)
add_lines(ax_top, lines_HE_II, "gold", "HeII lines", 500)
ax_top.legend(loc="upper right")
ax_top.set_ylim(0.3, 1.4)
ax_top.set_xlim(4000, 9000)
ax_top.set_xlabel("Wavelength (Å)")
ax_top.set_ylabel("Normalized Flux")
ax_top.set_title(f"{spec_files[i]} and temp {teffs[i]:.0f}")


ax_hb_main, ax_hb_res = make_zoom_column(gs[1, 1], H_BETA, 200, r"H$\beta$")
ax_ha_main, ax_ha_res = make_zoom_column(gs[1, 2], H_ALPHA, 200, r"H$\alpha$")
ax_hei_main, ax_hei_res = make_zoom_column(gs[1, 0], HE_I_4471, 60, "He I 4471")

ax_hb_main.set_ylabel("Normalized Flux")

connect_zoom(ax_top, ax_hb_main, H_BETA - 200, H_BETA + 200)
connect_zoom(ax_top, ax_ha_main, H_ALPHA - 200, H_ALPHA + 200)
connect_zoom(ax_top, ax_hei_main, HE_I_4471 - 60, HE_I_4471 + 60)


plt.show()

In [ ]:
hello = (ew > 5.0) & (ew_errs < 1.0) & (bof < 2)
hellu = np.where(mask)[0]

# sort best_indices by ew, highest first
hellu = hellu[np.argsort(ew[hellu])[::-1]]
print(len(hellu))
print(hellu[:200])

In [ ]:
mask = (ew > 5.0) & (ew_errs < 1.0) & (bof < 2)
best_indices = np.where(mask)[0]

# sort best_indices by ew, highest first
best_indices = best_indices[np.argsort(ew[best_indices])[::-1]]

print(len(best_indices))
print(best_indices[:10])          # top 10 indices

In [ ]:
ew_he_4471 = spectra["nana_HeI_4471_EW"].to_numpy()
ew_he_4471_err = spectra["nana_HeI_4471_EW_err"].to_numpy()

mask = (bof < 1) & (teffs > 24000) & (ew_he_4471_err < 2)
valid_indices = np.where(mask)[0]

# sort valid indices by EW, descending
sorted_by_ew = valid_indices[np.argsort(ew_he_4471[valid_indices])]#[::-1]]

top_n = 10
top_10_he = sorted_by_ew[:top_n]

print(top_10_he)
print(ew_he_4471[top_10_he])

In [ ]:
mask = (ew > 1) & (bof < 2) & (np.isnan(linewidth) == False)
valid_indices = np.where(mask)[0]

#sorted by linewidth
sorted_by_lw = valid_indices[np.argsort(linewidth[valid_indices])] #[::-1]]
print("HUH", linewidth[valid_indices])
print(sorted_by_lw[:10])
linewidth_indices = sorted_by_lw[:25]
print(linewidth_indices)

In [ ]:
print(f"Nan example, EW = {ew[20479]}, m1 = {m1[20479]} , m2 = {m2[20479]}, m3 = {m3[20479]}, m4 = {m4[20479]}")

In [ ]:
for rank, indx in enumerate(linewidth_indices):
    i = indx
    fig = plt.figure(figsize=(12, 8))
    gs = gridspec.GridSpec(2, 3, height_ratios=[1.3, 1.0], hspace=0.3, wspace=0.3)
    
    # ---------------- Top overview panel ----------------
    ax_top = fig.add_subplot(gs[0, :])
    ax_top.step(lam, data[i], c="k", where="mid", lw=0.75)
    ax_top.plot(lam, synth_spec[i], "r-", lw=0.75)
    
    stats_text = (
    f"Teff = {teffs[i]:.0f}K\n"
    f"EW(H$\\alpha$) = {ew[i]:.1f}\n"
    f"Centroid = {centroid[i]:.1f}\n"
    f"Linewidth = {linewidth[i]:.1f}\n"
    f"Skew = {skew[i]:.1f}\n"
    f"Kurtosis = {kurtosis[i]:.1f}\n"
    f"BOF = {bof[i]:.1f}")
    
    ax_top.text(
        0.87, 0.4, stats_text,
        transform=ax_top.transAxes,
        fontsize=10, va="top", ha="left",
        bbox=dict(boxstyle="round", facecolor="white", edgecolor="0.5", alpha=0.85),
    )
    
    add_lines(ax_top, lines_balmer, "palevioletred", "Balmer lines", 1000)
    add_lines(ax_top, lines_HE_I, "darkseagreen", "HeI lines", 500)
    add_lines(ax_top, lines_HE_II, "gold", "HeII lines", 500)
    ax_top.legend(loc="upper right")
    ax_top.set_ylim(0.3, 1.4)
    ax_top.set_xlim(4000, 9000)
    ax_top.set_xlabel("Wavelength (Å)")
    ax_top.set_ylabel("Normalized Flux")
    ax_top.set_title(f"{rank+1:04d}, GAIA ID: {gaia_ids[i]}, and {spec_files[i]}")
    
    
    
    ax_hb_main, ax_hb_res = make_zoom_column(gs[1, 1], H_BETA, 200, r"H$\beta$")
    ax_ha_main, ax_ha_res = make_zoom_column(gs[1, 2], H_ALPHA, 200, r"H$\alpha$")
    ax_hei_main, ax_hei_res = make_zoom_column(gs[1, 0], HE_I_4471, 60, "He I 4471")
    
    ax_hb_main.set_ylabel("Normalized Flux")
    
    connect_zoom(ax_top, ax_hb_main, H_BETA - 200, H_BETA + 200)
    connect_zoom(ax_top, ax_ha_main, H_ALPHA - 200, H_ALPHA + 200)
    connect_zoom(ax_top, ax_hei_main, HE_I_4471 - 60, HE_I_4471 + 60)
    #plt.savefig(f"{rank+1:04d}_{gaia_ids[i]}_spectrum")
    
    plt.show()

In [ ]:
#500 iterations into pdf

output_dir = "highest_ew_halpha"
os.makedirs(output_dir, exist_ok=True)


with PdfPages(os.path.join(output_dir, "pretty_spectra.pdf")) as pdf:
    for rank, indx in enumerate(best_indices):
        i = indx
        fig = plt.figure(figsize=(12, 8))
        gs = gridspec.GridSpec(2, 3, height_ratios=[1.3, 1.0], hspace=0.3, wspace=0.3)
        
        # ---------------- Top overview panel ----------------
        ax_top = fig.add_subplot(gs[0, :])
        ax_top.step(lam, data[i], c="k", where="mid", lw=0.75)
        ax_top.plot(lam, synth_spec[i], "r-", lw=0.75)
        
        stats_text = (
        f"Teff = {teffs[i]:.0f}K\n"
        f"EW(H$\\alpha$) = {ew[i]:.1f}\n"
        f"Centroid = {centroid[i]:.1f}\n"
        f"Linewidth = {linewidth[i]:.1f}\n"
        f"Skew = {skew[i]:.1f}\n"
        f"Kurtosis = {kurtosis[i]:.1f}\n"
        f"BOF = {bof[i]:.1f}")
        
        ax_top.text(
            0.87, 0.4, stats_text,
            transform=ax_top.transAxes,
            fontsize=10, va="top", ha="left",
            bbox=dict(boxstyle="round", facecolor="white", edgecolor="0.5", alpha=0.85),
        )
        
        add_lines(ax_top, lines_balmer, "palevioletred", "Balmer lines", 1000)
        add_lines(ax_top, lines_HE_I, "darkseagreen", "HeI lines", 500)
        add_lines(ax_top, lines_HE_II, "gold", "HeII lines", 500)
        ax_top.legend(loc="upper right")
        ax_top.set_ylim(0.3, 1.4)
        ax_top.set_xlim(4000, 9000)
        ax_top.set_xlabel("Wavelength (Å)")
        ax_top.set_ylabel("Normalized Flux")
        ax_top.set_title(f"{rank+1:04d}, GAIA ID: {gaia_ids[i]}, and {spec_files[i]}")
        
        
        
        ax_hb_main, ax_hb_res = make_zoom_column(gs[1, 1], H_BETA, 200, r"H$\beta$")
        ax_ha_main, ax_ha_res = make_zoom_column(gs[1, 2], H_ALPHA, 200, r"H$\alpha$")
        ax_hei_main, ax_hei_res = make_zoom_column(gs[1, 0], HE_I_4471, 60, "He I 4471")
        
        ax_hb_main.set_ylabel("Normalized Flux")
        
        connect_zoom(ax_top, ax_hb_main, H_BETA - 200, H_BETA + 200)
        connect_zoom(ax_top, ax_ha_main, H_ALPHA - 200, H_ALPHA + 200)
        connect_zoom(ax_top, ax_hei_main, HE_I_4471 - 60, HE_I_4471 + 60)
        #fig.savefig(os.path.join(output_dir, f"{rank+1:04d}_{gaia_ids[i]}.png"), dpi=150, bbox_inches="tight")
        pdf.savefig(fig)
        plt.close(fig)
print("Done!")

In [ ]:
#25 highest and lowest linewidth

output_dir = "highest_linewidth"
output_dir = "lowest_linewidth"

os.makedirs(output_dir, exist_ok=True)

with PdfPages(os.path.join(output_dir, "low_linewidth_spectra.pdf")) as pdf:
    for rank, indx in enumerate(linewidth_indices):
        i = indx
        fig = plt.figure(figsize=(12, 8))
        gs = gridspec.GridSpec(2, 3, height_ratios=[1.3, 1.0], hspace=0.3, wspace=0.3)
        
        # ---------------- Top overview panel ----------------
        ax_top = fig.add_subplot(gs[0, :])
        ax_top.step(lam, data[i], c="k", where="mid", lw=0.75)
        ax_top.plot(lam, synth_spec[i], "r-", lw=0.75)
        
        stats_text = (
        f"Teff = {teffs[i]:.0f}K\n"
        f"EW(H$\\alpha$) = {ew[i]:.1f}\n"
        f"Centroid = {centroid[i]:.1f}\n"
        f"Linewidth = {linewidth[i]:.1f}\n"
        f"Skew = {skew[i]:.1f}\n"
        f"Kurtosis = {kurtosis[i]:.1f}\n"
        f"BOF = {bof[i]:.1f}")
        
        ax_top.text(
            0.87, 0.4, stats_text,
            transform=ax_top.transAxes,
            fontsize=10, va="top", ha="left",
            bbox=dict(boxstyle="round", facecolor="white", edgecolor="0.5", alpha=0.85),
        )
        
        add_lines(ax_top, lines_balmer, "palevioletred", "Balmer lines", 1000)
        add_lines(ax_top, lines_HE_I, "darkseagreen", "HeI lines", 500)
        add_lines(ax_top, lines_HE_II, "gold", "HeII lines", 500)
        ax_top.legend(loc="upper right")
        ax_top.set_ylim(0.3, 1.4)
        ax_top.set_xlim(4000, 9000)
        ax_top.set_xlabel("Wavelength (Å)")
        ax_top.set_ylabel("Normalized Flux")
        ax_top.set_title(f"{rank+1:04d}, GAIA ID: {gaia_ids[i]}, and {spec_files[i]}")
        
        
        
        ax_hb_main, ax_hb_res = make_zoom_column(gs[1, 1], H_BETA, 200, r"H$\beta$")
        ax_ha_main, ax_ha_res = make_zoom_column(gs[1, 2], H_ALPHA, 200, r"H$\alpha$")
        ax_hei_main, ax_hei_res = make_zoom_column(gs[1, 0], HE_I_4471, 60, "He I 4471")
        
        ax_hb_main.set_ylabel("Normalized Flux")
        
        connect_zoom(ax_top, ax_hb_main, H_BETA - 200, H_BETA + 200)
        connect_zoom(ax_top, ax_ha_main, H_ALPHA - 200, H_ALPHA + 200)
        connect_zoom(ax_top, ax_hei_main, HE_I_4471 - 60, HE_I_4471 + 60)
        #fig.savefig(os.path.join(output_dir, f"{rank+1:04d}_{gaia_ids[i]}.png"), dpi=150, bbox_inches="tight")
        pdf.savefig(fig)
        plt.close(fig)
print("Done!")

In [ ]:
# Find ways to make scatter plots of x vs y colored by z such that 
# we can actually see tens of thousands of points (because I think our Be star sample will be 10000-ish?)

# Start by making Halpha EW vs Hbeta EW, colored by stellar Teff.

# Drop the error bars for now; or draw one cross which is the size of the median error bar of the plotted points.

# Also, can you see if the synthesis method we use from robusta returns some kind of likelihood value or loss value? I
# f it does, what is it? If it doesn’t, I’ll request it from Hilder.

In [ ]:
#
ews_ha = spectra["nana_Halpha_EW"].to_numpy()
ews_ha_err = spectra["nana_Halpha_EW_err"].to_numpy()

ews_hb = spectra["nana_Hbeta_EW"].to_numpy()
ews_hb_err = spectra["nana_Hbeta_EW_err"].to_numpy()

print(teffs.shape, ews_hb.shape)

In [ ]:
#Halpha EW vs Hbeta EW for stars with Halpha EW > 1. and Halpha EW err > 1. With error bars.

#try out the Hgamma and Hdelta, we don't expect emission

good = (bof < 1.8) & (ews_ha_err < 1.0)

be = (ews_ha > 1)
be_good = good & be
print("How many?", be_good.sum())
median_ha_err = np.median(ews_ha_err[be_good])
median_hb_err = np.median(ews_hb_err[be_good])


f = plt.figure(figsize=(8, 5))
plt.xlabel("Halpha EW")
plt.ylabel("Hbeta EW")
plt.title("Halpha EW vs Hbeta EW good Be(?) stars (small sample)")
plt.axhline(0, alpha = 0.3, color = "blue")
plt.ylim(-1, 5)
print(mask.sum())
sc = plt.scatter(ews_ha[be_good], ews_hb[be_good], alpha = 0.3, s = 7, c = teffs[be_good], cmap = "gist_rainbow", edgecolor = "black", linewidth = 1, marker = "s")
plt.colorbar(sc, label = "Temperature (K)")
colors = sc.to_rgba(teffs[be_good])
plt.scatter(14, 4.2, color = "red", s = 20, edgecolor = "black", label = "median error bar", zorder = 20)
plt.errorbar(14, 4.2, xerr = median_ha_err, yerr = median_hb_err, zorder = 2, ecolor = "red")
plt.errorbar(ews_ha[be_good], ews_hb[be_good], xerr = ews_ha_err[be_good], yerr = ews_hb_err[be_good], fmt = "none", ecolor = colors, markersize = 2, alpha = 0.3, zorder = 5)
#plt.errorbar(ews_ha[be_good], ews_hb[be_good], xerr = ews_hb_err[be_good], yerr = ews_ha_err[be_good], fmt = "none", ecolor = "k", elinewidth = 1, markersize = 2, alpha = 0.5, zorder = 1)
plt.xlim(0,15)
plt.legend()
plt.show()

In [ ]:
#histogram of linewidth

plt.hist(ew_errs, bins = 100)
plt.title("Histogram of linewidth (24k spectra sample)")
plt.semilogy()